# 07 樹種分類・K/L材分類可能性評価（900-1700 nm）

近赤外スペクトルのうち900-1700 nm相当の範囲だけを使い、樹種ごとおよびK/L材で分類可能性があるかを可視化・診断し、軽量な分類モデルで検証します。

主な流れ:

1. データ読み込み、樹種ラベル、K/L材ラベル作成
2. 樹種分類・K/L材分類の可能性診断
3. 樹種ペアごとの分離可能性評価
4. 前処理探索
5. 900-1700 nm分類モデル探索
6. 波長区間分類モデル探索
7. valid確認、混同行列、分類レポート
8. 混同行列・分類レポートによる分類可能性の整理

K/L材分類は、現時点では `conifer -> K`, `broadleaf -> L` として定義しています。逆の場合は `KL_LABEL_MAPPING` を入れ替えてください。

この07では `SPECTRAL_RANGE_NM = (900, 1700)` とし、データに存在する範囲との交差だけを特徴量に使います。今回のデータは約1001-2500 nm相当なので、実際の使用範囲は約1001-1700 nmです。

## 0. 設定と共通関数

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path('outputs/.matplotlib_cache').resolve()))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from scipy.signal import savgol_filter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)

JAPANESE_FONT_CANDIDATES = [
    'Meiryo',
    'Meiriyo',
    'Hiragino Sans',
    'Hiragino Maru Gothic Pro',
    'Hiragino Mincho ProN',
    'Yu Gothic',
    'YuGothic',
    'Noto Sans CJK JP',
    'Noto Sans JP',
    'IPAexGothic',
    'TakaoGothic',
    'Osaka',
]
AVAILABLE_FONTS = {f.name for f in font_manager.fontManager.ttflist}
JAPANESE_FONT = next((font for font in JAPANESE_FONT_CANDIDATES if font in AVAILABLE_FONTS), None)
if JAPANESE_FONT is None:
    JAPANESE_FONT = 'DejaVu Sans'

PLOT_RC = {
    'font.family': 'sans-serif',
    'font.sans-serif': [JAPANESE_FONT],
    'figure.figsize': (10, 5),
    'axes.unicode_minus': False,
}
sns.set_theme(style='whitegrid', rc=PLOT_RC)
plt.rcParams.update(PLOT_RC)
print(f'Using plot font: {JAPANESE_FONT}')

DATA_DIR = Path('../data')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SAMPLE_SUBMIT_PATH = DATA_DIR / 'sample_submit.csv'

# 07では900-1700 nm相当だけを使います。データが波数(cm-1)表記の場合は nm = 1e7 / cm-1 で判定します。
SPECTRAL_RANGE_NM = (900.0, 1700.0)


PREFIX = '07_species_kl_classification_900_1700nm'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('outputs/production/07_species_kl_classification_900_1700nm') / RUN_ID
FIGURE_DIR = OUTPUT_DIR / 'figures'
for d in [OUTPUT_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SEARCH_MODE = 'quick'  # 'quick', 'standard', 'exhaustive'
RUN_PAIRWISE_DIAGNOSTICS = True
RUN_PREPROCESSING_SEARCH = True
RUN_FULL_WAVELENGTH_SEARCH = True
RUN_INTERVAL_SEARCH = True

# 同じノートで2つの分類タスクを流します。必要に応じて片方だけにしてください。
CLASSIFICATION_TASKS = ['species', 'kl']  # 'species', 'kl'
PRIMARY_METRIC = 'balanced_accuracy'
VALID_SIZE = 0.20
TEST_SIZE = 0.15
N_JOBS = -1

SAMPLE_CANDIDATES = ['sample number', 'sample_id', 'id', 'ID', 'No', 'no']
SPECIES_NUMBER_CANDIDATES = ['species number', 'species_number', 'species_id', '樹種番号']
SPECIES_NAME_CANDIDATES = ['樹種', 'species', 'species_name']
GROUP_CANDIDATES = ['ロット', 'lot', 'batch', 'sample_name']

# K/L材分類の根拠になる分類。イチョウは裸子植物のため針葉樹側に置きます。
WOOD_TYPE_MAPPING = {
    1: 'conifer',      # イチョウ
    3: 'broadleaf',   # ウエンジ
    4: 'broadleaf',   # ウォールナット
    5: 'broadleaf',   # クリ
    8: 'conifer',     # スプルース
    11: 'broadleaf',  # チェリー
    12: 'broadleaf',  # トチ
    13: 'broadleaf',  # ナラ
    14: 'conifer',    # ヒノキ
    15: 'conifer',    # ベイスギ
    16: 'conifer',    # 米ヒバ
    17: 'conifer',    # ベイマツ
    19: 'broadleaf',  # ホワイトオーク
}
KL_LABEL_MAPPING = {
    'conifer': 'K',
    'broadleaf': 'L',
}

SEARCH_PROFILES = {
    'quick': {
        'cv_splits': 3,
        'top_preprocessings': 4,
        'top_models_for_interval': 4,
        'max_interval_candidates': 24,
        'top_models_for_valid': 8,
        'top_models_for_ensemble': 4,
        'run_group_cv': True,
    },
    'standard': {
        'cv_splits': 5,
        'top_preprocessings': 6,
        'top_models_for_interval': 8,
        'max_interval_candidates': 48,
        'top_models_for_valid': 15,
        'top_models_for_ensemble': 6,
        'run_group_cv': True,
    },
    'exhaustive': {
        'cv_splits': 5,
        'top_preprocessings': 8,
        'top_models_for_interval': 15,
        'max_interval_candidates': 90,
        'top_models_for_valid': 30,
        'top_models_for_ensemble': 8,
        'run_group_cv': True,
    },
}
PROFILE = SEARCH_PROFILES[SEARCH_MODE]


def read_csv_with_fallback(path: Path) -> pd.DataFrame:
    encodings = ['utf-8-sig', 'utf-8', 'cp932', 'shift_jis']
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def pick_first_existing(columns, candidates, required=False, label='column'):
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise ValueError(f'{label} not found. candidates={candidates}')
    return None


def get_spectral_columns(df: pd.DataFrame, exclude_cols: list[str]) -> list[str]:
    spectral_cols = []
    for col in df.columns:
        if col in exclude_cols:
            continue
        try:
            float(col)
            spectral_cols.append(col)
        except (TypeError, ValueError):
            continue
    return spectral_cols


def infer_spectral_axis_unit(axis: np.ndarray) -> str:
    return 'cm-1' if np.nanmedian(axis) > 2500 else 'nm'


def spectral_axis_to_nm(axis: np.ndarray, unit: str) -> np.ndarray:
    axis = np.asarray(axis, dtype=float)
    if unit == 'cm-1':
        return 1e7 / axis
    return axis


def select_spectral_columns_by_nm(spectral_cols, axis_values, unit, nm_range):
    if nm_range is None:
        nm_values = spectral_axis_to_nm(axis_values, unit)
        return spectral_cols, axis_values, nm_values
    lo, hi = nm_range
    nm_values = spectral_axis_to_nm(axis_values, unit)
    mask = (nm_values >= lo) & (nm_values <= hi)
    selected_cols = [c for c, keep in zip(spectral_cols, mask) if keep]
    selected_axis = np.array([float(c) for c in selected_cols], dtype=float)
    selected_nm = spectral_axis_to_nm(selected_axis, unit)
    if not selected_cols:
        raise ValueError(f'No spectral columns found in nm range {nm_range}. Data nm range={np.nanmin(nm_values):.1f}-{np.nanmax(nm_values):.1f}')
    return selected_cols, selected_axis, selected_nm


def format_spectral_axis(ax, unit: str):
    ax.set_xlabel('Wavelength (nm)' if unit == 'nm' else 'Wavenumber (cm$^{-1}$)')
    if unit == 'cm-1':
        ax.invert_xaxis()


def make_stratified_split(df, label_col, valid_size=VALID_SIZE, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    data = df.dropna(subset=[label_col]).reset_index(drop=True).copy()
    y = data[label_col].astype(str)
    train_valid, test = train_test_split(
        data,
        test_size=test_size,
        random_state=random_state,
        stratify=y if y.value_counts().min() >= 2 else None,
    )
    y_tv = train_valid[label_col].astype(str)
    valid_ratio = valid_size / (1 - test_size)
    train, valid = train_test_split(
        train_valid,
        test_size=valid_ratio,
        random_state=random_state,
        stratify=y_tv if y_tv.value_counts().min() >= 2 else None,
    )
    return train.reset_index(drop=True), valid.reset_index(drop=True), test.reset_index(drop=True)


def write_json(path: Path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

run_config = {
    'run_id': RUN_ID,
    'search_mode': SEARCH_MODE,
    'classification_tasks': CLASSIFICATION_TASKS,
    'primary_metric': PRIMARY_METRIC,
    'spectral_range_nm': SPECTRAL_RANGE_NM,
    'wood_type_mapping': WOOD_TYPE_MAPPING,
    'kl_label_mapping': KL_LABEL_MAPPING,
    'profile': PROFILE,
}
write_json(OUTPUT_DIR / f'{PREFIX}_run_config.json', run_config)
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

## 1. データ読み込みと分類ラベル作成

In [ ]:
df = read_csv_with_fallback(TRAIN_PATH)
test_df_raw = read_csv_with_fallback(TEST_PATH) if TEST_PATH.exists() else None
sample_submit_df = read_csv_with_fallback(SAMPLE_SUBMIT_PATH) if SAMPLE_SUBMIT_PATH.exists() else None

species_number_col = pick_first_existing(df.columns, SPECIES_NUMBER_CANDIDATES, required=True, label='species number column')
species_name_col = pick_first_existing(df.columns, SPECIES_NAME_CANDIDATES, required=False, label='species name column')
sample_col = pick_first_existing(df.columns, SAMPLE_CANDIDATES, required=False, label='sample id column')
group_col = pick_first_existing(df.columns, GROUP_CANDIDATES, required=False, label='group column')
exclude_cols = [c for c in [species_number_col, species_name_col, sample_col, group_col] if c is not None]
all_spectral_cols = get_spectral_columns(df, exclude_cols=exclude_cols)
all_axis_values = np.array([float(c) for c in all_spectral_cols], dtype=float)
spectral_axis_unit = infer_spectral_axis_unit(all_axis_values)

if not all_spectral_cols:
    raise ValueError('No spectral columns detected. Spectral columns must have numeric column names.')

spectral_cols, wavelengths, spectral_nm = select_spectral_columns_by_nm(
    all_spectral_cols,
    all_axis_values,
    spectral_axis_unit,
    SPECTRAL_RANGE_NM,
)
all_spectral_nm = spectral_axis_to_nm(all_axis_values, spectral_axis_unit)
spectral_range_df = pd.DataFrame({
    'setting': ['all_detected', 'selected_for_07'],
    'n_features': [len(all_spectral_cols), len(spectral_cols)],
    'axis_min': [float(np.nanmin(all_axis_values)), float(np.nanmin(wavelengths))],
    'axis_max': [float(np.nanmax(all_axis_values)), float(np.nanmax(wavelengths))],
    'nm_min': [float(np.nanmin(all_spectral_nm)), float(np.nanmin(spectral_nm))],
    'nm_max': [float(np.nanmax(all_spectral_nm)), float(np.nanmax(spectral_nm))],
})

label_df = df.copy()
label_df['species_label'] = label_df[species_number_col].astype(int).astype(str)
if species_name_col is not None:
    label_df['species_display'] = label_df[species_number_col].astype(int).astype(str) + '_' + label_df[species_name_col].astype(str)
else:
    label_df['species_display'] = label_df['species_label']
label_df['wood_type'] = label_df[species_number_col].astype(int).map(WOOD_TYPE_MAPPING)
label_df['kl_label'] = label_df['wood_type'].map(KL_LABEL_MAPPING)

missing_wood_type = label_df.loc[label_df['wood_type'].isna(), species_number_col].drop_duplicates().tolist()
if missing_wood_type:
    raise ValueError(f'WOOD_TYPE_MAPPING is missing species numbers: {missing_wood_type}')

species_table = (
    label_df[[species_number_col, species_name_col, 'wood_type', 'kl_label']] if species_name_col else label_df[[species_number_col, 'wood_type', 'kl_label']]
).drop_duplicates().sort_values(species_number_col)
species_table.to_csv(OUTPUT_DIR / f'{PREFIX}_species_kl_mapping.csv', index=False, encoding='utf-8-sig')

task_label_cols = {'species': 'species_display', 'kl': 'kl_label'}

summary_rows = []
for task in CLASSIFICATION_TASKS:
    label_col = task_label_cols[task]
    counts = label_df[label_col].value_counts().rename_axis('class').reset_index(name='n')
    counts['task'] = task
    summary_rows.append(counts)
class_count_df = pd.concat(summary_rows, ignore_index=True)
class_count_df.to_csv(OUTPUT_DIR / f'{PREFIX}_class_counts.csv', index=False, encoding='utf-8-sig')
spectral_range_df.to_csv(OUTPUT_DIR / f'{PREFIX}_spectral_range_summary.csv', index=False, encoding='utf-8-sig')

print(f'n_samples={len(label_df)}, n_spectral_cols={len(spectral_cols)}, axis={wavelengths.min():.1f}-{wavelengths.max():.1f} {spectral_axis_unit}, nm={spectral_nm.min():.1f}-{spectral_nm.max():.1f}')
display(spectral_range_df)
display(species_table)
display(class_count_df)

## 2. 樹種・K/L材の分類可能性診断

モデル探索に入る前に、樹種別・K/L材別の平均スペクトル、PCA空間での分布、特徴量との関連を確認します。

In [ ]:
X_all = label_df[spectral_cols].astype(float)

class SavitzkyGolayTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, window_length=15, polyorder=2, deriv=0):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        n_features = arr.shape[1]
        win = min(self.window_length, n_features if n_features % 2 == 1 else n_features - 1)
        win = max(win, self.polyorder + 2 + (self.polyorder + 2) % 2)
        if win % 2 == 0:
            win -= 1
        if win <= self.polyorder:
            return arr
        return savgol_filter(arr, window_length=win, polyorder=self.polyorder, deriv=self.deriv, axis=1)

class SNVTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        mean = arr.mean(axis=1, keepdims=True)
        std = arr.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        return (arr - mean) / std

class MSCTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.reference_ = arr.mean(axis=0)
        return self
    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        corrected = np.empty_like(arr)
        ref = self.reference_
        for i, row in enumerate(arr):
            slope, intercept = np.polyfit(ref, row, 1)
            corrected[i] = (row - intercept) / slope if slope != 0 else row - intercept
        return corrected

class PLSDAClassifier(BaseEstimator):
    def __init__(self, n_components=5):
        self.n_components = n_components
    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder()
        y_enc = self.label_encoder_.fit_transform(y)
        Y = label_binarize(y_enc, classes=np.arange(len(self.label_encoder_.classes_)))
        if Y.shape[1] == 1:
            Y = np.column_stack([1 - Y[:, 0], Y[:, 0]])
        n_comp = min(self.n_components, np.asarray(X).shape[1], max(1, np.asarray(X).shape[0] - 1), Y.shape[1])
        self.model_ = PLSRegression(n_components=n_comp)
        self.model_.fit(X, Y)
        return self
    def predict(self, X):
        scores = self.model_.predict(X)
        idx = np.argmax(scores, axis=1)
        if len(self.label_encoder_.classes_) == 2 and scores.shape[1] == 2:
            idx = np.argmax(scores, axis=1)
        idx = np.clip(idx, 0, len(self.label_encoder_.classes_) - 1)
        return self.label_encoder_.inverse_transform(idx)
    def decision_function(self, X):
        return self.model_.predict(X)
    def predict_proba(self, X):
        z = self.model_.predict(X)
        z = z - z.max(axis=1, keepdims=True)
        exp = np.exp(z)
        return exp / exp.sum(axis=1, keepdims=True)

PREPROCESSORS = {
    'raw': [],
    'center_scale': [StandardScaler()],
    'savgol_smooth': [SavitzkyGolayTransformer(window_length=15, polyorder=2, deriv=0), StandardScaler()],
    'savgol_1st': [SavitzkyGolayTransformer(window_length=15, polyorder=2, deriv=1), StandardScaler()],
    'savgol_2nd': [SavitzkyGolayTransformer(window_length=15, polyorder=2, deriv=2), StandardScaler()],
    'snv': [SNVTransformer(), StandardScaler()],
    'snv_savgol_1st': [SNVTransformer(), SavitzkyGolayTransformer(window_length=15, polyorder=2, deriv=1), StandardScaler()],
    'snv_savgol_2nd': [SNVTransformer(), SavitzkyGolayTransformer(window_length=15, polyorder=2, deriv=2), StandardScaler()],
    'msc': [MSCTransformer(), StandardScaler()],
    'msc_savgol_2nd': [MSCTransformer(), SavitzkyGolayTransformer(window_length=15, polyorder=2, deriv=2), StandardScaler()],
}


def apply_preprocessor(name, X):
    Xt = X.copy()
    for step in PREPROCESSORS[name]:
        Xt = step.fit_transform(Xt)
    return np.asarray(Xt, dtype=float)


def plot_mean_spectra_by_label(label_col, prefix):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    for label, idx in label_df.groupby(label_col).groups.items():
        mean_raw = X_all.loc[idx].mean(axis=0).values
        axes[0].plot(wavelengths, mean_raw, linewidth=1.2, label=str(label))
    axes[0].set_title(f'Mean raw spectra by {label_col}')
    axes[0].set_ylabel('Signal')
    format_spectral_axis(axes[0], spectral_axis_unit)
    axes[0].legend(fontsize=7, ncol=2)

    X_snv = apply_preprocessor('snv_savgol_2nd', X_all)
    for label, idx in label_df.groupby(label_col).groups.items():
        mean_proc = X_snv[list(idx)].mean(axis=0)
        axes[1].plot(wavelengths, mean_proc, linewidth=1.2, label=str(label))
    axes[1].set_title(f'Mean SNV + 2nd derivative spectra by {label_col}')
    axes[1].set_ylabel('Processed signal')
    format_spectral_axis(axes[1], spectral_axis_unit)
    axes[1].legend(fontsize=7, ncol=2)
    fig.savefig(FIGURE_DIR / f'{PREFIX}_{prefix}_mean_spectra.png', dpi=160)
    plt.show()

for task in CLASSIFICATION_TASKS:
    label_col = task_label_cols[task]
    plot_mean_spectra_by_label(label_col, task)

# PCA score plot: raw SNV化後に確認します。
X_pca_input = apply_preprocessor('snv', X_all)
pca = PCA(n_components=min(8, X_pca_input.shape[0], X_pca_input.shape[1]), random_state=RANDOM_STATE)
scores = pca.fit_transform(X_pca_input)
pca_df = pd.DataFrame(scores[:, :2], columns=['PC1', 'PC2'])
pca_df['species'] = label_df['species_display'].values
pca_df['kl'] = label_df['kl_label'].values
pca_df.to_csv(OUTPUT_DIR / f'{PREFIX}_pca_scores.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='species', s=32, alpha=0.85, ax=axes[0])
axes[0].set_title('PCA score plot by species')
axes[0].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='kl', s=42, alpha=0.85, ax=axes[1])
axes[1].set_title('PCA score plot by K/L')
for ax in axes:
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)')
fig.savefig(FIGURE_DIR / f'{PREFIX}_pca_score_species_kl.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. 樹種ペア・K/L材の分離可能性評価

軽量な分類器で、どの樹種ペアが分けやすいか、どのペアが混ざりやすいかを確認します。

In [ ]:
def build_binary_pipeline(preprocess_name='snv_savgol_2nd'):
    steps = [(f'prep_{i}', step) for i, step in enumerate(PREPROCESSORS[preprocess_name])]
    steps.append(('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=RANDOM_STATE)))
    return Pipeline(steps)

pairwise_rows = []
if RUN_PAIRWISE_DIAGNOSTICS:
    species_labels = sorted(label_df['species_display'].unique())
    for a, b in itertools.combinations(species_labels, 2):
        sub = label_df[label_df['species_display'].isin([a, b])].copy()
        if sub['species_display'].value_counts().min() < PROFILE['cv_splits']:
            continue
        X_sub = sub[spectral_cols].astype(float)
        y_sub = sub['species_display'].astype(str)
        cv = StratifiedKFold(n_splits=PROFILE['cv_splits'], shuffle=True, random_state=RANDOM_STATE)
        pipe = build_binary_pipeline('snv_savgol_2nd')
        scores = cross_validate(
            pipe, X_sub, y_sub, cv=cv,
            scoring={'balanced_accuracy': 'balanced_accuracy', 'f1_macro': 'f1_macro'},
            n_jobs=N_JOBS,
            error_score=np.nan,
        )
        pairwise_rows.append({
            'class_a': a,
            'class_b': b,
            'n_a': int((y_sub == a).sum()),
            'n_b': int((y_sub == b).sum()),
            'balanced_accuracy_mean': float(np.nanmean(scores['test_balanced_accuracy'])),
            'f1_macro_mean': float(np.nanmean(scores['test_f1_macro'])),
        })

pairwise_df = pd.DataFrame(pairwise_rows).sort_values('balanced_accuracy_mean', ascending=True) if pairwise_rows else pd.DataFrame()
pairwise_df.to_csv(OUTPUT_DIR / f'{PREFIX}_species_pairwise_separability.csv', index=False, encoding='utf-8-sig')
if not pairwise_df.empty:
    display(pairwise_df.head(20))
    display(pairwise_df.tail(20))

    heat = pairwise_df.pivot(index='class_a', columns='class_b', values='balanced_accuracy_mean')
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(heat, cmap='viridis', vmin=0.5, vmax=1.0, annot=False, ax=ax)
    ax.set_title('Pairwise species separability: balanced accuracy')
    fig.savefig(FIGURE_DIR / f'{PREFIX}_species_pairwise_separability_heatmap.png', dpi=160, bbox_inches='tight')
    plt.show()

# K/Lのクラス構成と樹種内訳
kl_species_table = pd.crosstab(label_df['species_display'], label_df['kl_label'])
kl_species_table.to_csv(OUTPUT_DIR / f'{PREFIX}_kl_by_species_crosstab.csv', encoding='utf-8-sig')
display(kl_species_table)

## 4. 前処理探索

分類タスクごとに、まず軽量な基準モデルで前処理の効き方を見ます。

In [ ]:
def make_cv(y, n_splits=None):
    n_splits = n_splits or PROFILE['cv_splits']
    counts = pd.Series(y).value_counts()
    feasible = max(2, min(n_splits, int(counts.min())))
    return StratifiedKFold(n_splits=feasible, shuffle=True, random_state=RANDOM_STATE)


def classifier_candidates(search_mode='quick'):
    base = {
        'plsda_3': PLSDAClassifier(n_components=3),
        'plsda_6': PLSDAClassifier(n_components=6),
        'logreg_l2_C1': LogisticRegression(max_iter=4000, C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
        'linear_svc_C1': LinearSVC(C=1.0, class_weight='balanced', random_state=RANDOM_STATE),
        'rbf_svc_C3_gamma_scale': SVC(C=3.0, gamma='scale', kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
        'knn_5_distance': KNeighborsClassifier(n_neighbors=5, weights='distance'),
        'random_forest_300': RandomForestClassifier(n_estimators=300, max_features='sqrt', class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS),
        'extra_trees_300': ExtraTreesClassifier(n_estimators=300, max_features='sqrt', class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS),
    }
    if search_mode in ['standard', 'exhaustive']:
        base.update({
            'logreg_l2_C0.3': LogisticRegression(max_iter=4000, C=0.3, class_weight='balanced', random_state=RANDOM_STATE),
            'logreg_l2_C3': LogisticRegression(max_iter=4000, C=3.0, class_weight='balanced', random_state=RANDOM_STATE),
            'linear_svc_C0.3': LinearSVC(C=0.3, class_weight='balanced', random_state=RANDOM_STATE),
            'rbf_svc_C1_gamma_scale': SVC(C=1.0, gamma='scale', kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
            'knn_11_distance': KNeighborsClassifier(n_neighbors=11, weights='distance'),
            'decision_tree_balanced': DecisionTreeClassifier(max_depth=None, min_samples_leaf=3, class_weight='balanced', random_state=RANDOM_STATE),
        })
    if search_mode == 'exhaustive':
        base.update({
            'plsda_10': PLSDAClassifier(n_components=10),
            'rbf_svc_C10_gamma_scale': SVC(C=10.0, gamma='scale', kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
            'random_forest_600': RandomForestClassifier(n_estimators=600, max_features='sqrt', class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=N_JOBS),
            'extra_trees_600': ExtraTreesClassifier(n_estimators=600, max_features='sqrt', class_weight='balanced', random_state=RANDOM_STATE, n_jobs=N_JOBS),
        })
    return base


def build_pipeline(preprocess_name, model):
    steps = [(f'prep_{i}', step) for i, step in enumerate(PREPROCESSORS[preprocess_name])]
    steps.append(('model', model))
    return Pipeline(steps)


def evaluate_pipeline_cv(pipe, X, y, cv):
    scoring = {
        'accuracy': 'accuracy',
        'balanced_accuracy': 'balanced_accuracy',
        'f1_macro': 'f1_macro',
        'f1_weighted': 'f1_weighted',
    }
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=N_JOBS, error_score=np.nan)
    return {f'{k}_mean': float(np.nanmean(v)) for k, v in scores.items() if k.startswith('test_')}

preprocessing_scores = []
if RUN_PREPROCESSING_SEARCH:
    for task in CLASSIFICATION_TASKS:
        y = label_df[task_label_cols[task]].astype(str)
        X = label_df[spectral_cols].astype(float)
        cv = make_cv(y)
        base_model = LogisticRegression(max_iter=4000, C=1.0, class_weight='balanced', random_state=RANDOM_STATE)
        for prep_name in PREPROCESSORS:
            pipe = build_pipeline(prep_name, base_model)
            result = evaluate_pipeline_cv(pipe, X, y, cv)
            result.update({'task': task, 'preprocess': prep_name, 'model': 'logreg_l2_C1', 'cv_splits': cv.get_n_splits()})
            preprocessing_scores.append(result)

preprocessing_scores_df = pd.DataFrame(preprocessing_scores)
if not preprocessing_scores_df.empty:
    preprocessing_scores_df = preprocessing_scores_df.sort_values(['task', 'test_balanced_accuracy_mean'], ascending=[True, False])
    preprocessing_scores_df.to_csv(OUTPUT_DIR / f'{PREFIX}_preprocessing_scores.csv', index=False, encoding='utf-8-sig')
    display(preprocessing_scores_df)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=preprocessing_scores_df, x='preprocess', y='test_balanced_accuracy_mean', hue='task', ax=ax)
    ax.set_title('Preprocessing search: balanced accuracy')
    ax.set_xlabel('Preprocessing')
    ax.set_ylabel('CV balanced accuracy')
    ax.tick_params(axis='x', rotation=45)
    fig.savefig(FIGURE_DIR / f'{PREFIX}_preprocessing_search.png', dpi=160, bbox_inches='tight')
    plt.show()

def top_preprocessings_for_task(task):
    if preprocessing_scores_df.empty:
        return ['raw', 'snv', 'snv_savgol_2nd']
    sub = preprocessing_scores_df[preprocessing_scores_df['task'] == task].copy()
    return sub.head(PROFILE['top_preprocessings'])['preprocess'].tolist()

## 5. 900-1700 nm分類モデル探索

前処理探索で上位になった前処理を使い、900-1700 nm相当のスペクトル範囲だけで分類モデルを探索します。

In [ ]:
full_search_rows = []
full_search_predictions = []
model_dict = classifier_candidates(SEARCH_MODE)

if RUN_FULL_WAVELENGTH_SEARCH:
    for task in CLASSIFICATION_TASKS:
        y = label_df[task_label_cols[task]].astype(str)
        X = label_df[spectral_cols].astype(float)
        cv = make_cv(y)
        for prep_name in top_preprocessings_for_task(task):
            for model_name, model in model_dict.items():
                pipe = build_pipeline(prep_name, model)
                result = evaluate_pipeline_cv(pipe, X, y, cv)
                result.update({
                    'task': task,
                    'preprocess': prep_name,
                    'model': model_name,
                    'interval_name': 'full',
                    'n_features': len(spectral_cols),
                    'cv_splits': cv.get_n_splits(),
                })
                full_search_rows.append(result)

full_search_df = pd.DataFrame(full_search_rows)
if not full_search_df.empty:
    full_search_df = full_search_df.sort_values(['task', 'test_balanced_accuracy_mean'], ascending=[True, False])
    full_search_df.to_csv(OUTPUT_DIR / f'{PREFIX}_full_wavelength_model_scores.csv', index=False, encoding='utf-8-sig')
    display(full_search_df.groupby('task').head(15))

    fig, axes = plt.subplots(len(CLASSIFICATION_TASKS), 1, figsize=(13, 4 * len(CLASSIFICATION_TASKS)), constrained_layout=True)
    if len(CLASSIFICATION_TASKS) == 1:
        axes = [axes]
    for ax, task in zip(axes, CLASSIFICATION_TASKS):
        plot_df = full_search_df[full_search_df['task'] == task].head(20).copy()
        plot_df['name'] = plot_df['preprocess'] + ' / ' + plot_df['model']
        sns.barplot(data=plot_df, y='name', x='test_balanced_accuracy_mean', ax=ax)
        ax.set_title(f'Full wavelength model search: {task}')
        ax.set_xlabel('CV balanced accuracy')
        ax.set_ylabel('')
    fig.savefig(FIGURE_DIR / f'{PREFIX}_full_wavelength_model_scores.png', dpi=160, bbox_inches='tight')
    plt.show()

## 6. 波長区間選択

細かく刻みすぎず、中〜広めの単一区間と、2〜3区間の組み合わせを評価します。実際に評価する候補リストはCSV保存し、ここで表示します。

In [ ]:
def rounded_step(value):
    if value <= 100:
        return 50
    if value <= 250:
        return 100
    return 250


def interval_mask(intervals):
    mask = np.zeros_like(wavelengths, dtype=bool)
    for lo, hi in intervals:
        a, b = min(lo, hi), max(lo, hi)
        mask |= (wavelengths >= a) & (wavelengths <= b)
    return mask


def make_interval_name(intervals):
    return '+'.join([f'{int(round(min(a,b)))}-{int(round(max(a,b)))}' for a, b in intervals])


def build_interval_candidates(max_candidates=None):
    axis_min, axis_max = float(np.nanmin(wavelengths)), float(np.nanmax(wavelengths))
    span = axis_max - axis_min
    step = rounded_step(span / 30)
    widths = sorted(set([round(span * r / step) * step for r in [0.10, 0.15, 0.22, 0.30, 0.40]]))
    widths = [w for w in widths if w > 0 and w < span * 0.95]

    rows = []
    seen = set()

    def add(intervals, kind):
        clean = tuple((float(min(a, b)), float(max(a, b))) for a, b in intervals)
        mask = interval_mask(clean)
        if mask.sum() < 5:
            return
        key = tuple((round(a, 3), round(b, 3)) for a, b in clean)
        if key in seen:
            return
        seen.add(key)
        rows.append({
            'interval_name': make_interval_name(clean),
            'intervals': clean,
            'interval_kind': kind,
            'combo_size': len(clean),
            'n_features': int(mask.sum()),
        })

    domain_intervals = [(4000, 5000), (5000, 6000), (6000, 7500), (7500, 9000), (9000, 10000)] if spectral_axis_unit == 'cm-1' else [(900, 1200), (1200, 1400), (1400, 1600), (1600, 1800), (1000, 1600)]
    for iv in domain_intervals:
        add([iv], 'single_domain')

    for width in widths:
        starts = np.arange(math.floor(axis_min / step) * step, axis_max - width + step, step * 2)
        for st in starts:
            add([(st, st + width)], 'single_sliding')

    explicit_combos = [
        [(4000, 5000), (6000, 7500)],
        [(5000, 6000), (7500, 9000)],
        [(4000, 5000), (6000, 7500), (9000, 10000)],
        [(5200, 7200), (7000, 9000)],
    ] if spectral_axis_unit == 'cm-1' else [
        [(1000, 1600), (1200, 1400)],
        [(900, 1200), (1300, 1700)],
        [(1000, 1200), (1400, 1600)],
        [(1100, 1300), (1450, 1600)],
    ]
    for combo in explicit_combos:
        add(combo, 'combo_explicit')

    representative = [r['intervals'][0] for r in rows if r['combo_size'] == 1]
    representative = representative[:18]
    for size in [2, 3]:
        for combo in itertools.combinations(representative, size):
            centers = [(a + b) / 2 for a, b in combo]
            if min(np.diff(sorted(centers))) < span * 0.12:
                continue
            add(list(combo), f'combo_{size}_regions')
            if max_candidates and len(rows) >= max_candidates:
                break
        if max_candidates and len(rows) >= max_candidates:
            break

    out = pd.DataFrame(rows)
    if max_candidates:
        priority = {'single_domain': 0, 'combo_explicit': 1, 'combo_2_regions': 2, 'combo_3_regions': 3, 'single_sliding': 4}
        out['priority'] = out['interval_kind'].map(priority).fillna(9)
        out = out.sort_values(['priority', 'combo_size', 'n_features'], ascending=[True, True, False]).head(max_candidates).drop(columns=['priority'])
    return out.reset_index(drop=True)

interval_candidates_df = build_interval_candidates(PROFILE['max_interval_candidates'])
interval_candidates_df.assign(intervals_text=interval_candidates_df['intervals'].astype(str)).drop(columns=['intervals']).to_csv(
    OUTPUT_DIR / f'{PREFIX}_interval_candidates.csv', index=False, encoding='utf-8-sig'
)
display(interval_candidates_df.assign(intervals=interval_candidates_df['intervals'].astype(str)))

interval_rows = []
if RUN_INTERVAL_SEARCH and not full_search_df.empty:
    for task in CLASSIFICATION_TASKS:
        y = label_df[task_label_cols[task]].astype(str)
        X_base = label_df[spectral_cols].astype(float)
        cv = make_cv(y)
        top_full = full_search_df[full_search_df['task'] == task].head(PROFILE['top_models_for_interval'])
        for _, top_row in top_full.iterrows():
            prep_name = top_row['preprocess']
            model_name = top_row['model']
            model = model_dict[model_name]
            for _, iv_row in interval_candidates_df.iterrows():
                mask = interval_mask(iv_row['intervals'])
                cols = [c for c, m in zip(spectral_cols, mask) if m]
                if len(cols) < 5:
                    continue
                X = X_base[cols]
                pipe = build_pipeline(prep_name, model)
                result = evaluate_pipeline_cv(pipe, X, y, cv)
                result.update({
                    'task': task,
                    'preprocess': prep_name,
                    'model': model_name,
                    'interval_name': iv_row['interval_name'],
                    'interval_kind': iv_row['interval_kind'],
                    'combo_size': int(iv_row['combo_size']),
                    'n_features': len(cols),
                    'cv_splits': cv.get_n_splits(),
                })
                interval_rows.append(result)

interval_scores_df = pd.DataFrame(interval_rows)
if not interval_scores_df.empty:
    interval_scores_df = interval_scores_df.sort_values(['task', 'test_balanced_accuracy_mean'], ascending=[True, False])
    interval_scores_df.to_csv(OUTPUT_DIR / f'{PREFIX}_interval_model_scores.csv', index=False, encoding='utf-8-sig')
    display(interval_scores_df.groupby('task').head(20))

## 7. valid確認・混同行列・分類レポート

CV上位モデルをtrainで学習し、validで確認します。目的は最終モデル化ではなく、どの程度分類できそうか、どのクラスが混同されるかを確認することです。

In [ ]:
def collect_candidate_rows(task):
    frames = []
    if not full_search_df.empty:
        frames.append(full_search_df[full_search_df['task'] == task].copy())
    if not interval_scores_df.empty:
        frames.append(interval_scores_df[interval_scores_df['task'] == task].copy())
    if not frames:
        return pd.DataFrame()
    cand = pd.concat(frames, ignore_index=True)
    return cand.sort_values('test_balanced_accuracy_mean', ascending=False).head(PROFILE['top_models_for_valid']).reset_index(drop=True)

valid_rows = []
fitted_models = {}

for task in CLASSIFICATION_TASKS:
    label_col = task_label_cols[task]
    train_df, valid_df, holdout_df = make_stratified_split(label_df, label_col)
    candidates = collect_candidate_rows(task)
    if candidates.empty:
        continue
    for i, row in candidates.iterrows():
        if row['interval_name'] == 'full':
            cols = spectral_cols
        else:
            interval_row = interval_candidates_df[interval_candidates_df['interval_name'] == row['interval_name']].iloc[0]
            mask = interval_mask(interval_row['intervals'])
            cols = [c for c, m in zip(spectral_cols, mask) if m]
        X_train = train_df[cols].astype(float)
        y_train = train_df[label_col].astype(str)
        X_valid = valid_df[cols].astype(float)
        y_valid = valid_df[label_col].astype(str)
        model = model_dict[row['model']]
        pipe = build_pipeline(row['preprocess'], model)
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_valid)
        result = {
            'task': task,
            'rank': i + 1,
            'preprocess': row['preprocess'],
            'model': row['model'],
            'interval_name': row['interval_name'],
            'n_features': len(cols),
            'cv_balanced_accuracy': row['test_balanced_accuracy_mean'],
            'valid_accuracy': accuracy_score(y_valid, pred),
            'valid_balanced_accuracy': balanced_accuracy_score(y_valid, pred),
            'valid_f1_macro': f1_score(y_valid, pred, average='macro'),
            'valid_f1_weighted': f1_score(y_valid, pred, average='weighted'),
        }
        if task == 'kl' and hasattr(pipe, 'predict_proba'):
            try:
                proba = pipe.predict_proba(X_valid)
                if proba.shape[1] == 2:
                    result['valid_roc_auc'] = roc_auc_score(y_valid, proba[:, 1])
            except Exception:
                result['valid_roc_auc'] = np.nan
        valid_rows.append(result)
        fitted_models[(task, i + 1)] = {'pipeline': pipe, 'cols': cols, 'label_col': label_col, 'valid_df': valid_df, 'holdout_df': holdout_df}

valid_scores_df = pd.DataFrame(valid_rows)
if not valid_scores_df.empty:
    valid_scores_df = valid_scores_df.sort_values(['task', 'valid_balanced_accuracy'], ascending=[True, False])
    valid_scores_df.to_csv(OUTPUT_DIR / f'{PREFIX}_valid_scores.csv', index=False, encoding='utf-8-sig')
    display(valid_scores_df)

    for task in CLASSIFICATION_TASKS:
        sub = valid_scores_df[valid_scores_df['task'] == task]
        if sub.empty:
            continue
        best = sub.iloc[0]
        key = (task, int(best['rank']))
        item = fitted_models[key]
        pipe = item['pipeline']
        cols = item['cols']
        valid_df_task = item['valid_df']
        y_valid = valid_df_task[item['label_col']].astype(str)
        pred = pipe.predict(valid_df_task[cols].astype(float))
        labels = sorted(y_valid.unique())
        cm = confusion_matrix(y_valid, pred, labels=labels)
        cm_df = pd.DataFrame(cm, index=labels, columns=labels)
        cm_df.to_csv(OUTPUT_DIR / f'{PREFIX}_{task}_best_valid_confusion_matrix.csv', encoding='utf-8-sig')
        report = classification_report(y_valid, pred, output_dict=True, zero_division=0)
        pd.DataFrame(report).T.to_csv(OUTPUT_DIR / f'{PREFIX}_{task}_best_valid_classification_report.csv', encoding='utf-8-sig')

        fig, ax = plt.subplots(figsize=(9, 7))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f'Best valid confusion matrix: {task}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        fig.savefig(FIGURE_DIR / f'{PREFIX}_{task}_best_valid_confusion_matrix.png', dpi=160, bbox_inches='tight')
        plt.show()
        print(f'Best {task}: {best.to_dict()}')
        display(pd.DataFrame(report).T)

## 8. 分類可能性評価のまとめ

valid確認までの結果を、後で比較しやすい形でCSVにまとめます。アンサンブルや最終モデル保存は、このノートでは行いません。

In [ ]:
summary_tables = {}
if 'preprocessing_scores_df' in globals() and not preprocessing_scores_df.empty:
    summary_tables['preprocessing_top'] = preprocessing_scores_df.groupby('task').head(10)
if 'full_search_df' in globals() and not full_search_df.empty:
    summary_tables['full_wavelength_top'] = full_search_df.groupby('task').head(10)
if 'interval_scores_df' in globals() and not interval_scores_df.empty:
    summary_tables['interval_top'] = interval_scores_df.groupby('task').head(10)
if 'valid_scores_df' in globals() and not valid_scores_df.empty:
    summary_tables['valid_top'] = valid_scores_df.groupby('task').head(10)

for name, table in summary_tables.items():
    table.to_csv(OUTPUT_DIR / f'{PREFIX}_{name}_summary.csv', index=False, encoding='utf-8-sig')
    print(f'[{name}]')
    display(table)

print(f'Evaluation artifacts saved under: {OUTPUT_DIR}')


## 9. 07の読み方

- `*_species_kl_mapping.csv`: 樹種番号、樹種名、針葉樹/広葉樹、K/Lラベルの対応表
- `*_class_counts.csv`: 分類タスクごとのクラス件数
- `*_spectral_range_summary.csv`: 全検出スペクトル範囲と07で実際に使用した900-1700 nm交差範囲
- `*_species_pairwise_separability.csv`: 樹種ペアごとの分離しやすさ
- `*_preprocessing_scores.csv`: 前処理探索結果
- `*_full_wavelength_model_scores.csv`: 900-1700 nmモデル探索結果
- `*_interval_candidates.csv`: 実際に評価した波長区間候補
- `*_interval_model_scores.csv`: 波長区間モデル探索結果
- `*_valid_scores.csv`: valid確認結果
- `*_valid_top_summary.csv`: 分類可能性評価の主要結果

K/Lの定義を逆にしたい場合は、冒頭の `KL_LABEL_MAPPING` だけを入れ替えて再実行します。